In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [3]:
mean = (0.4914, 0.4822, 0.4465)
std  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

In [5]:
trainset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=train_transform
)
testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=test_transform
)

batch_size = 128
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
testloader  = DataLoader(testset,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print("Train size:", len(trainset), "Test size:", len(testset))

100%|██████████| 170M/170M [00:20<00:00, 8.13MB/s] 


Train size: 50000 Test size: 10000


In [7]:
images, labels = next(iter(trainloader))
print("Images:", images.shape, images.dtype)
print("Labels:", labels.shape, labels.dtype)
print("Label example:", labels[:10])

C:\Users\ben21\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Images: torch.Size([128, 3, 32, 32]) torch.float32
Labels: torch.Size([128]) torch.int64
Label example: tensor([4, 9, 3, 1, 6, 3, 3, 6, 9, 7])


In [9]:
class BasicCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # (B,32,32,32)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (B,32,16,16)

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # (B,64,16,16)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (B,64,8,8)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = BasicCNN().to(device)
print(model)

BasicCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)


In [11]:
images, labels = images.to(device), labels.to(device)
logits = model(images)
print("Logits shape:", logits.shape)


Logits shape: torch.Size([128, 10])


In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

loss = criterion(logits, labels)
print("One batch loss:", loss.item())


One batch loss: 2.315535306930542


In [26]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, total_correct / total

In [28]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, total_correct / total

In [30]:
epochs = 5
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, trainloader)
    test_loss, test_acc = evaluate(model, testloader)

    print(f"Epoch {epoch:02d}/{epochs} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | "
          f"Test Loss: {test_loss:.4f} Acc: {test_acc*100:.2f}%")

Epoch 01/5 | Train Loss: 1.3769 Acc: 51.16% | Test Loss: 1.0895 Acc: 61.81%
Epoch 02/5 | Train Loss: 1.1794 Acc: 57.86% | Test Loss: 1.0155 Acc: 64.13%
Epoch 03/5 | Train Loss: 1.0948 Acc: 61.28% | Test Loss: 0.9076 Acc: 68.26%
Epoch 04/5 | Train Loss: 1.0394 Acc: 63.12% | Test Loss: 0.8737 Acc: 69.30%
Epoch 05/5 | Train Loss: 0.9945 Acc: 64.89% | Test Loss: 0.8451 Acc: 70.64%
